# 04 | Otimização do modelo de Credit Scoring
### Lending Club Loan Data · Tech Challenge Fase 04 · Machine Learning Engineering

**Objetivo:** melhorar a identificação da classe inadimplente sem escolher configurações olhando o teste final. Este é o **quarto notebook interno**, não a Etapa 4 oficial do desafio.

O baseline V1 é preservado. Vamos comparar a Regressão Logística tradicional e a versão `class_weight="balanced"`, investigar thresholds na **validação**, escolher uma configuração por critério documentado e somente então fazer **uma avaliação final** no teste reservado.

**Pontos de controle:** carregamento eficiente → população e split idênticos à V1 → comparação na validação → seleção → retreino → teste final → persistência → recarga.

> **Limite metodológico:** o valor de Accuracy isoladamente não é suficiente para este problema (target ~80/20). Nosso objetivo experimental é priorizar Recall por meio de F2, impondo também uma precisão mínima ilustrativa. Este limite **não é uma política real de crédito**. A versão V1 já foi testada; a escolha da V2 será feita sem consultar suas métricas no teste.

## 0. Bibliotecas e configurações

In [ ]:
from pathlib import Path
import gc
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, fbeta_score, precision_score,
    recall_score, roc_auc_score, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

from credit_scoring_observability.preprocessing import (
    BASELINE_FEATURES, GOOD_STATUS, BAD_STATUS,
    engineer_features, split_features_target, split_train_test,
)

In [ ]:
COLORS = {
    "navy": "#193551", "blue": "#397CF4", "teal": "#00A6A6",
    "coral": "#F46A6A", "purple": "#7A60EF", "gold": "#EAAF38",
    "green": "#28AA78", "gray": "#6E7581",
}
plt.rcParams.update({
    "figure.figsize": (9, 5), "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.18,
    "axes.titleweight": "bold", "axes.titlesize": 14,
})
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
CHUNK_SIZE = 25_000
TRAIN_SAMPLE_SIZE = 300_000
VALIDATION_FRACTION = 0.20
MIN_VALIDATION_PRECISION = 0.30  # regra experimental, não decisão de negócio
THRESHOLDS = np.round(np.arange(0.10, 0.901, 0.025), 3)

## 1. Caminhos e integridade dos arquivos

Os dados permanecem locais na pasta `data/` ignorada pelo Git. Nenhum arquivo do baseline V1 será sobrescrito.

In [ ]:
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_METRICS_PATH = MODELS_DIR / "baseline_metrics.json"
OPTIMIZED_MODEL_PATH = MODELS_DIR / "credit_scoring_optimized.joblib"
OPTIMIZED_METADATA_PATH = MODELS_DIR / "credit_scoring_optimized_metadata.json"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset não encontrado: {DATA_PATH.resolve()}")

print("Dataset localizado:", DATA_PATH.resolve())
print("Baseline V1 será preservado.")

## 2. Carregamento otimizado da população de modelagem

A leitura repete o notebook 03: **todos os registros** são percorridos em chunks e apenas os empréstimos com desfecho conhecido permanecem. As variáveis numéricas são convertidas para `float32`, reduzindo o custo de memória. `term` e `emp_length` não são convertidas para `category` para manter compatibilidade com `engineer_features()`.

In [ ]:
cols_to_load = BASELINE_FEATURES + ["loan_status", "issue_d"]
frames = []
for chunk in pd.read_csv(
    DATA_PATH, usecols=cols_to_load, chunksize=CHUNK_SIZE, low_memory=True
):
    filtered = chunk.loc[
        chunk["loan_status"].isin(GOOD_STATUS + BAD_STATUS)
    ].copy()
    float_cols = filtered.select_dtypes(include=["float64"]).columns
    filtered[float_cols] = filtered[float_cols].astype("float32")
    frames.append(filtered)

df = pd.concat(frames, ignore_index=True)
del frames

group_cols = [
    "home_ownership", "verification_status", "purpose",
    "application_type", "loan_status",
]
for col in group_cols:
    df[col] = df[col].astype("category")

df["target"] = df["loan_status"].isin(BAD_STATUS).astype("int8")
gc.collect()
print("Dimensões:", df.shape)
print(f"Memória: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
print(f"Taxa de inadimplência: {df['target'].mean():.2%}")

## 3. Feature engineering e split idênticos à V1

In [ ]:
# Proteção extra para o mapeamento de emp_length no módulo existente.
df["term"] = df["term"].astype("object")
df["emp_length"] = df["emp_length"].astype("object")
df = engineer_features(df)
X, y, metadata = split_features_target(df)
X_train_full, X_test, y_train_full, y_test = split_train_test(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Treino original:", X_train_full.shape)
print("Teste reservado:", X_test.shape)
print("Proporção positiva treino/teste:", round(y_train_full.mean(), 4), round(y_test.mean(), 4))
assert len(X_test) == 269_620, "Verifique se a base e o split são os mesmos da V1."

### 3.1 Amostra de treinamento de 300 mil registros

Repetimos a **mesma regra e semente da V1**, para que a comparação entre modelos seja interpretável. Os outros registros do treino não entram no ajuste desta rodada; o teste completo segue intocado.

In [ ]:
train_size = min(TRAIN_SAMPLE_SIZE, len(X_train_full))
X_train_model, X_unused, y_train_model, y_unused = train_test_split(
    X_train_full, y_train_full,
    train_size=train_size, random_state=RANDOM_STATE, stratify=y_train_full
)
del X_unused, y_unused, X_train_full, y_train_full, X, y, metadata, df
gc.collect()

X_train_model = X_train_model.copy()
X_test = X_test.copy()
numeric_features = X_train_model.select_dtypes(include="number").columns.tolist()
categorical_features = [c for c in X_train_model.columns if c not in numeric_features]

for col in numeric_features:
    X_train_model[col] = X_train_model[col].astype("float32")
    X_test[col] = X_test[col].astype("float32")
for col in categorical_features:
    X_train_model[col] = X_train_model[col].astype("object")
    X_test[col] = X_test[col].astype("object")

print("Amostra treino:", X_train_model.shape)
print("Teste completo:", X_test.shape)
print("Features numéricas/categóricas:", len(numeric_features), len(categorical_features))
print(y_train_model.value_counts(normalize=True))

## 4. Treino interno × validação

O teste final **não participará** do ajuste de thresholds ou da seleção da configuração. Reservamos 20% dos 300 mil registros da amostra de treino para avaliação de alternativas.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_model, y_train_model,
    test_size=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
    stratify=y_train_model,
)
print("Ajuste:", X_fit.shape)
print("Validação:", X_val.shape)
print("Teste fechado:", X_test.shape)
assert X_fit.index.intersection(X_val.index).empty
assert X_fit.index.intersection(X_test.index).empty
assert X_val.index.intersection(X_test.index).empty

## 5. Fábrica de pipelines independentes

Utilizamos `copy=True` no imputador e no escalonador para **não alterar os DataFrames recebidos durante a predição**, corrigindo o comportamento que observamos no teste de recarga da V1. O One-Hot permanece esparso.

In [ ]:
def make_pipeline(class_weight=None):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", copy=True)),
        ("scaler", RobustScaler(copy=True)),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore", sparse_output=True, dtype=np.float32
        )),
    ])
    processor = ColumnTransformer(
        [("numeric", num_pipe, numeric_features),
         ("categorical", cat_pipe, categorical_features)],
        sparse_threshold=1.0,
    )
    model = LogisticRegression(
        solver="saga", max_iter=500, tol=1e-3,
        class_weight=class_weight, random_state=RANDOM_STATE,
    )
    return Pipeline([("preprocessor", processor), ("model", model)])

print("Pipelines prontos: tradicional e class_weight='balanced'.")

## 6. Experimentos na validação

Treinamos **uma configuração por vez**, armazenamos apenas as probabilidades de validação e liberamos o modelo temporário. Assim reduzimos o pico de memória da execução.

> Um `ConvergenceWarning` será mostrado e registrado. Não o escondemos nem interpretamos um ajuste sem convergência como resultado definitivo.

In [ ]:
configurations = {"standard": None, "balanced": "balanced"}
validation_probabilities = {}
fit_log = []

for config_name, weight in configurations.items():
    print(f"\nTreinando: {config_name}")
    temp_model = make_pipeline(class_weight=weight)
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as seen:
        warnings.simplefilter("always", ConvergenceWarning)
        temp_model.fit(X_fit, y_fit)
    proba = temp_model.predict_proba(X_val.copy(deep=True))[:, 1]
    convergence_notes = [str(w.message) for w in seen if issubclass(w.category, ConvergenceWarning)]
    validation_probabilities[config_name] = proba
    fit_log.append({
        "config": config_name,
        "minutes": (time.perf_counter() - t0) / 60,
        "roc_auc_val": roc_auc_score(y_val, proba),
        "average_precision_val": average_precision_score(y_val, proba),
        "convergence_warning": bool(convergence_notes),
    })
    print("Tempo (min):", round(fit_log[-1]["minutes"], 2))
    print("ROC-AUC/AP:", round(fit_log[-1]["roc_auc_val"], 4),
          round(fit_log[-1]["average_precision_val"], 4))
    if convergence_notes:
        print("AVISO DE CONVERGÊNCIA:", convergence_notes[0])
    del temp_model
    gc.collect()

pd.DataFrame(fit_log).set_index("config")

## 7. Estudo dos thresholds

Para cada configuração, testamos thresholds de `0,10` a `0,90`, observando **Precision, Recall, F1, F2** e proporção de previsões positivas. O F2 dá peso maior ao Recall; a restrição de Precision evita escolher um ponto que simplesmente sinalize quase todos os clientes como inadimplentes.

In [ ]:
results = []
for config_name, proba in validation_probabilities.items():
    for thr in THRESHOLDS:
        y_val_pred = (proba >= thr).astype(np.int8)
        results.append({
            "config": config_name,
            "threshold": float(thr),
            "precision": precision_score(y_val, y_val_pred, zero_division=0),
            "recall": recall_score(y_val, y_val_pred, zero_division=0),
            "f1": f1_score(y_val, y_val_pred, zero_division=0),
            "f2": fbeta_score(y_val, y_val_pred, beta=2, zero_division=0),
            "predicted_positive_rate": float(y_val_pred.mean()),
        })
threshold_results = pd.DataFrame(results)
threshold_results.sort_values("f2", ascending=False).head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for config, color in [("standard", COLORS["blue"]), ("balanced", COLORS["coral"])]:
    s = threshold_results.query("config == @config").sort_values("threshold")
    ax.plot(s["threshold"], s["recall"], marker="o", markersize=3,
            linewidth=2, label=f"Recall · {config}", color=color)
    ax.plot(s["threshold"], s["precision"], linestyle="--", linewidth=2,
            label=f"Precision · {config}", color=color)
ax.axhline(MIN_VALIDATION_PRECISION, color=COLORS["gray"],
           linestyle=":", label="Precision mínima ilustrativa")
ax.set(title="Precision e Recall por threshold (validação)",
       xlabel="Threshold", ylabel="Score", ylim=(0, 1))
ax.legend(ncol=2, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for config, color in [("standard", COLORS["blue"]), ("balanced", COLORS["purple"])]:
    s = threshold_results.query("config == @config").sort_values("threshold")
    ax.plot(s["threshold"], s["f2"], marker="o", markersize=3,
            linewidth=2.5, label=f"F2 · {config}", color=color)
ax.set(title="F2-Score por threshold (somente validação)",
       xlabel="Threshold", ylabel="F2", ylim=(0, 1))
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 8. Seleção transparente da configuração

**Critério experimental pré-definido:** entre os pares modelo/threshold que atingem Precision de pelo menos **0,30 na validação**, selecionar aquele com maior F2. Empates são resolvidos por Recall e, por último, por Precision. O valor 0,30 é uma referência ilustrativa para esta análise, **não** uma restrição regulatória ou política de crédito.

Se nenhuma configuração atender ao critério, interrompemos o notebook para revisar a estratégia, em vez de declarar automaticamente uma melhoria.

In [ ]:
eligible = threshold_results.loc[
    threshold_results["precision"] >= MIN_VALIDATION_PRECISION
].copy()
if eligible.empty:
    raise ValueError(
        "Nenhum candidato atingiu a Precision mínima na validação. "
        "Revise o limite/estratégia e registre a alteração antes do retreino."
    )
best = eligible.sort_values(
    ["f2", "recall", "precision"], ascending=[False, False, False]
).iloc[0]
SELECTED_CONFIG = str(best["config"])
SELECTED_THRESHOLD = float(best["threshold"])
SELECTED_CLASS_WEIGHT = configurations[SELECTED_CONFIG]
print("Configuração selecionada:", SELECTED_CONFIG)
print("Threshold validado:", SELECTED_THRESHOLD)
print("Métricas da validação:")
display(best.to_frame(name="resultado"))

## 9. Retreinar a configuração selecionada

Os 60 mil registros de validação retornam ao **conjunto de 300 mil registros**. Esta é a única configuração que será levada ao teste final. O pipeline completo inclui imputação, scaling, encoding e o classificador.

In [ ]:
del X_fit, X_val, y_fit, y_val
validation_probabilities.clear()
gc.collect()

optimized_pipeline = make_pipeline(class_weight=SELECTED_CLASS_WEIGHT)
start_final = time.perf_counter()
with warnings.catch_warnings(record=True) as seen_final:
    warnings.simplefilter("always", ConvergenceWarning)
    optimized_pipeline.fit(X_train_model, y_train_model)
training_minutes = (time.perf_counter() - start_final) / 60
final_convergence = [str(w.message) for w in seen_final if issubclass(w.category, ConvergenceWarning)]
print(f"Retreino final concluído em {training_minutes:.2f} minuto(s).")
if final_convergence:
    print("ATENÇÃO: ConvergenceWarning no treino final:", final_convergence[0])
else:
    print("Sem aviso de convergência.")

## 10. Avaliação ÚNICA no teste completo

A configuração e o threshold já foram escolhidos sem usar o teste. **Não altere parâmetros em resposta a estas métricas**: isso invalidaria a separação de avaliação.

In [ ]:
t0 = time.perf_counter()
# O pipeline foi definido com copy=True: o teste não deve sofrer mutação.
proba_test = optimized_pipeline.predict_proba(X_test.copy(deep=True))[:, 1]
pred_test = (proba_test >= SELECTED_THRESHOLD).astype(np.int8)
print(f"Predição concluída em {time.perf_counter() - t0:.2f}s; {len(y_test):,} casos.")
assert len(pred_test) == len(y_test)

In [ ]:
optimized_metrics = {
    "accuracy": accuracy_score(y_test, pred_test),
    "precision": precision_score(y_test, pred_test, zero_division=0),
    "recall": recall_score(y_test, pred_test, zero_division=0),
    "f1": f1_score(y_test, pred_test, zero_division=0),
    "f2": fbeta_score(y_test, pred_test, beta=2, zero_division=0),
    "roc_auc": roc_auc_score(y_test, proba_test),
    "pr_auc": average_precision_score(y_test, proba_test),
    "threshold": SELECTED_THRESHOLD,
}
metrics_v2_df = pd.Series(optimized_metrics, name="V2").to_frame()
display(metrics_v2_df)
print(classification_report(
    y_test, pred_test,
    target_names=["Não inadimplente", "Inadimplente"],
    digits=4, zero_division=0,
))

## 11. Matriz de confusão e tipos de erro

In [ ]:
cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["Não inadimplente", "Inadimplente"])
ax.set_yticks([0, 1], ["Não inadimplente", "Inadimplente"])
ax.set(xlabel="Previsto", ylabel="Real", title="Matriz de confusão · V2")
for (r, c), val in np.ndenumerate(cm):
    ax.text(c, r, f"{val:,}", ha="center", va="center",
            fontsize=12, fontweight="bold",
            color="white" if val > cm.max()/2 else COLORS["navy"])
fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04)
plt.tight_layout(); plt.show()
print("VN:", tn, "FP:", fp, "FN:", fn, "VP:", tp)

**Leitura:** falso negativo é um empréstimo inadimplente que o sistema classificou como não inadimplente. Falso positivo é um empréstimo pago sinalizado como inadimplente. O custo relativo dessas decisões depende da política da instituição e não pode ser inferido apenas a partir desta matriz.

## 12. Curvas ROC e Precision-Recall

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba_test)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba_test)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(fpr, tpr, color=COLORS["purple"], linewidth=2.5,
        label=f"V2 (AUC={optimized_metrics['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color=COLORS["gray"], label="Aleatório")
ax.set(xlabel="Taxa de falsos positivos", ylabel="Taxa de verdadeiros positivos",
       title="Curva ROC · Modelo V2")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(recall_curve, precision_curve, color=COLORS["coral"], linewidth=2.5,
        label=f"V2 (AP={optimized_metrics['pr_auc']:.3f})")
ax.axhline(y_test.mean(), color=COLORS["gray"], linestyle="--",
           label=f"Prevalência={y_test.mean():.3f}")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall · Modelo V2")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 13. Distribuição das probabilidades e threshold selecionado

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(proba_test[y_test.to_numpy()==0], bins=45, density=True, alpha=.57,
        color=COLORS["teal"], label="Não inadimplente")
ax.hist(proba_test[y_test.to_numpy()==1], bins=45, density=True, alpha=.57,
        color=COLORS["coral"], label="Inadimplente")
ax.axvline(SELECTED_THRESHOLD, color=COLORS["navy"], linestyle="--",
           linewidth=2, label=f"Threshold = {SELECTED_THRESHOLD:.3f}")
ax.set(xlabel="Probabilidade estimada de inadimplência", ylabel="Densidade",
       title="Distribuição das probabilidades · V2")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 14. Comparação descritiva V1 × V2

A V1 foi previamente avaliada no teste. Aqui **apenas relatamos** seus resultados salvos para contextualizar a V2; eles não participaram da seleção da configuração. Se `baseline_metrics.json` não estiver disponível, a V2 ainda poderá ser concluída normalmente.

In [ ]:
if BASELINE_METRICS_PATH.exists():
    with BASELINE_METRICS_PATH.open("r", encoding="utf-8") as f:
        baseline_metrics = json.load(f)
    shared_metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
    comparison = pd.DataFrame({
        "V1 (threshold 0,50)": [baseline_metrics.get(k, np.nan) for k in shared_metrics],
        f"V2 (threshold {SELECTED_THRESHOLD:.3f})": [optimized_metrics[k] for k in shared_metrics],
    }, index=shared_metrics)
    display(comparison)
    comparison.plot(kind="bar", figsize=(10, 5),
                    color=[COLORS["blue"], COLORS["green"]], rot=25)
    plt.title("Métricas observadas · V1 e V2")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.tight_layout(); plt.show()
else:
    print("baseline_metrics.json não encontrado; métricas V2 disponíveis acima.")

## 15. Persistência do modelo e do threshold

O arquivo `.joblib` salva o pipeline (API igual à V1). O JSON salva a configuração e **o threshold selecionado**. Atenção: `pipeline.predict(X)` usa o limite interno padrão de 0,50. Para reproduzir a V2, utilize `pipeline.predict_proba(X)[:, 1] >= threshold_do_JSON`.

Os arquivos V1 **não são sobrescritos**.

In [ ]:
if final_convergence:
    print("ATENÇÃO: o modelo final registrou ConvergenceWarning. Revise antes de publicar como versão definitiva.")

model_metadata = {
    "version": "V2_optimized",
    "dataset": "wordsforthewise/lending-club",
    "positive_class": 1,
    "random_state": RANDOM_STATE,
    "train_sample_rows": int(len(X_train_model)),
    "test_rows": int(len(X_test)),
    "validation_fraction": VALIDATION_FRACTION,
    "selection_criterion": f"max F2 subject to precision >= {MIN_VALIDATION_PRECISION} on validation",
    "selected_model": SELECTED_CONFIG,
    "class_weight": SELECTED_CLASS_WEIGHT,
    "decision_threshold": SELECTED_THRESHOLD,
    "feature_columns": list(X_train_model.columns),
    "metrics_on_test": {k: float(v) for k, v in optimized_metrics.items()},
    "convergence_warning": bool(final_convergence),
    "usage_note": "Use predict_proba(X)[:,1] >= decision_threshold; pipeline.predict uses threshold 0.5.",
}
joblib.dump(optimized_pipeline, OPTIMIZED_MODEL_PATH)
with OPTIMIZED_METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2, ensure_ascii=False)
print("Modelo salvo:", OPTIMIZED_MODEL_PATH)
print("Metadados salvos:", OPTIMIZED_METADATA_PATH)

## 16. Teste de recarga sem modificar a entrada

In [ ]:
reloaded_pipeline = joblib.load(OPTIMIZED_MODEL_PATH)
with OPTIMIZED_METADATA_PATH.open("r", encoding="utf-8") as f:
    reloaded_metadata = json.load(f)

sample = X_test.head(5).copy(deep=True)
original_prob = optimized_pipeline.predict_proba(sample.copy(deep=True))[:, 1]
reloaded_prob = reloaded_pipeline.predict_proba(sample.copy(deep=True))[:, 1]
is_equivalent = bool(np.allclose(original_prob, reloaded_prob, rtol=1e-8, atol=1e-10))
reloaded_threshold = float(reloaded_metadata["decision_threshold"])

verification = pd.DataFrame({
    "prob_original": original_prob,
    "prob_recarregada": reloaded_prob,
    "diferença_absoluta": np.abs(original_prob-reloaded_prob),
    "classe_v2": (reloaded_prob >= reloaded_threshold).astype(int),
})
display(verification)
assert is_equivalent, "Pipeline recarregado gerou probabilidades diferentes."
assert reloaded_threshold == SELECTED_THRESHOLD
assert OPTIMIZED_MODEL_PATH.exists() and OPTIMIZED_METADATA_PATH.exists()
print("Modelo + threshold salvos e recuperados com sucesso.")

## 17. Conclusão e handoff

O modelo otimizado foi selecionado **na validação**, ajustado novamente na amostra estratificada completa e avaliado uma vez no teste reservado. O relatório mantém explícitos Accuracy, Precision, Recall, F1, F2, ROC-AUC e Average Precision, sem confundir melhora no Recall com melhora automática da discriminação.

**Para os colegas:** utilizar as mesmas 23 features após `engineer_features()`, manter a ordem indicada em `feature_columns`, carregar `credit_scoring_optimized.joblib` e aplicar `decision_threshold` do JSON sobre `predict_proba(... )[:,1]`.

**Próximas entregas da Fabi:** revisar os outputs, criar o Data Contract com Pandera, demonstrar rejeição do `bad_batch` e preparar o handoff. O uso em concessão real requer governança, calibração, validação adicional e acompanhamento de vieses.

> **Execução:** escolha o kernel da `.venv`, use *Restart Kernel* antes do *Run All* e execute sequencialmente. Se surgir `MemoryError`, não aumente indiscriminadamente o uso de RAM: envie o erro e ajustaremos o tamanho do treino ou a estratégia de processamento mantendo o teste reservado.